Building a semantic search engine for real estate listings. Instead of only matching exact words like a keyword search, will use embeddings to understand the meaning of the listing descriptions to retrieve similar listings. 

Overview 
- Implement embedding-based semantic search using sentence-transformers. Build FAISS index for fast similarity search. Compare semantic vs keyword matching quality. 

Key Deliverables 
- Sentence embeddings for all listing remarks (384 or 768 dims) ✅
- FAISS index for efficient similarity search ✅
- SemanticSearcher class with query embedding + retrieval ✅ 
- Comparison study: semantic vs BM25 keyword search ✅
- Latency < 100ms for 10k listings ✅
- Relevance evaluation on 50 query-result pairs ✅

In [138]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Success")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3913.73it/s]


Success


In [38]:
from sentence_transformers import SentenceTransformer 
import faiss 
import numpy as np 
import pandas as pd
from rank_bm25 import BM25Okapi
import sys
import os
import json
import mysql.connector 
from dotenv import load_dotenv
import time
pd.reset_option("display.max_colwidth")

In [140]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.w3_entity_extractor import EntityExtractor

In [17]:
project_root = os.path.abspath('../')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from scripts.w2_text_cleaning import TextCleaner

========== REMARKS PROFILE ==========

Total listings: 1000
Null rate: 0.00%
Average length: 1290.09 characters

Price mentions: 57
Measurement mentions: 261
Room dimensions: 7

HTML tags: 0
Unicode usages: 394
Smart quotes: 276
Whitespace issues: 482
APN Counts: 3
Phone Number Counts: 0

Known abbreviations
ft       251
sq       237
condo    139
hoa      134
adu      122
bbq      104
rv       70
ev       58
hvac     57
ac       45

Unknown abbreviations
altos    3
sony     3
ages     3
wings    3
pulls    3
hemet    3
harte    3
rents    3
sj       3
tools    3
finds    3
items    3
lofts    3
gpm      2
draws    2
verde    2
lies     2
rises    2
tells    2
baja     2

Top 20 words
home         2383
living       1964
room         1233
offers       1073
space        1068
kitchen      953
private      849
bedroom      832
s            824
dining       815
2            796
features     761
you          750
area         690
located      668
spacious     667
bedrooms     666
new          

Note: FAISS is a library that allows us to quickly search through large collections of vectorss to find the embeddings closest to a user's query embedding.

In [26]:
class SemanticSearcher: 
    def __init__(self): 
        self.model = SentenceTransformer('all-MiniLM-L6-v2') 
        self.index = None 
        self.listings = None 
    def build_index(self, remarks_list): 
        print(f"Encoding {len(remarks_list)} listings...") 
        embeddings = self.model.encode(remarks_list) 
        # Build FAISS index 
        dim = embeddings.shape[1] 
        self.index = faiss.IndexFlatIP(dim)  # Inner product for cosine sim 
        faiss.normalize_L2(embeddings) 
        self.index.add(embeddings) 
        self.listings = remarks_list 
    def search(self, query, top_k=10): 
        query_emb = self.model.encode([query]) 
        faiss.normalize_L2(query_emb) 
        scores, indices = self.index.search(query_emb, top_k) 
        results = [(self.listings[i], scores[0][j]) for j, i in 
        enumerate(indices[0])] 
        return results 

In [142]:
df = pd.read_csv("../data/processed/cleaned_listing_sample.csv")
remarks = df['cleaned_remarks'].fillna("").tolist()

In [143]:
#pd.set_option("display.max_colwidth", None)
df['cleaned_remarks'].head()

0    This unique property offers two homes on one l...
1    Beautiful Two-Story Home in Victorville - Spac...
2    Presenting this exquisite second-floor corner ...
3    Thoughtfully renovated coastal retreat tucked ...
4    Welcome to 2085 Westhampton Drive in Arroyo Gr...
Name: cleaned_remarks, dtype: str

In [144]:
searcher = SemanticSearcher()
searcher.build_index(remarks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4278.59it/s]


Encoding 1000 listings...


In [145]:
embeddings = model.encode(remarks)

In [146]:
embeddings.shape

(1000, 384)

Each listing has a 384-dimensional embedding

In [147]:
# Saving the generated embeddings
#np.save("../data/listing_embeddings.npy", embeddings)

In [148]:
embeddings = np.load("../data/listing_embeddings.npy")

For deliverable 2, we have to make the embeddings made searchable

In [149]:
searcher = SemanticSearcher()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3225.35it/s]


In [150]:
searcher.build_index(remarks)

Encoding 1000 listings...


In [151]:
print(searcher.index.ntotal)

1000


The FAISS index now contains 1000 listing embeddings

In [152]:
print(searcher.index.d)

384


The index was built for 384-dimensional embeddings.

Deliverable 2 is completed

Deliverable 3 is where we basically implement the SemanticSearch search() method.

Just to note, this is how we are using the class: 

Cleaned MLS remarks

        |
        v

Convert remarks into embeddings

        |
        v

Store embeddings in FAISS

        |
        v

Accept a user's search query

        |
        v

Convert query into embedding

        |
        v
        
Find similar listings

Example

In [153]:
df['cleaned_remarks'].iloc[0]

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [154]:
results = searcher.search(
    df['cleaned_remarks'].iloc[0],
    top_k=5
)

In [155]:
df['cleaned_remarks'].iloc[0]

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [156]:
for remark, score in results:
    print("Similarity score:",score)
    print(remark)
    print('-------')

Similarity score: 1.0
This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!
-------
Similarity score: 0.75226396
Investment opportunity! The main home offers three bedrooms and two bathrooms. The property also includes additional converted living areas, bringing the total to six bedrooms, four bathrooms, and three kitchen areas. Featuring three separate entrances, the layout offers flexibility for multi-generational living or potential rental income. The home is fully fenced with parking and access from both 40th Avenue (front) and Rosedale (rear). Buyer to investigate and verify permit status, bedroom and bathroom count, and square footage o

In [157]:
query = "luxury kitchen with modern appliances"

results = searcher.search(query, top_k=5)

In [158]:
results

[('Modern Luxury Meets Elevated Design in this stunning Toll Brothers home located in the desirable Westridge collection within the gated Metropolitan Heights community. Positioned on a premium homesite, this three-story residence showcases sleek contemporary architecture, panoramic views from every level, and over $400000 in premium upgrades and enhancements. Step inside to experience an open-concept floor plan featuring dramatic floating stairs with glass railing, expansive living spaces, and an abundance of natural light. The interior has been beautifully upgraded with custom tile flooring downstairs and real wood flooring throughout the home, creating a warm yet sophisticated feel. The chef\'s kitchen is a true showpiece with premium JennAir appliances including a 48" gas range, built-in refrigerator, oversized island with Quartzite Blue Tahoe countertops, modern white acrylic cabinetry, and a stylish full-height backsplash. Designed for both comfort and entertaining, the spacious 

So far:

We have implemented the SemanticSearcher class using SentenceTransformer embeddings and FAISS retrieval. User queries are converted into 384-dimensional embeddings using all-MiniLM-L6-v2, normalized, and searched against the FAISS index. The system returns the top-k most semantically similar listing remarks ranked by cosine similarity.

Deliverable 4: Designing an experiment to compare two search approaches:
- Semantic Search: SentenceTransformer embeddings + FAISS that find listings based on the meaning and context of the remarks themselves 
- BM25 Keyword Search: Traditional information retrieval algorithm that find listings based on word overlap/frequency

In [27]:
class BM25Searcher:
    def __init__(self):
        self.bm25 = None
        self.documents = None

    def build_index(self, remarks_list):
        tokenized_docs = [
            remark.lower().split()
            for remark in remarks_list
        ]
        self.bm25 = BM25Okapi(tokenized_docs)
        self.documents = remarks_list

    def search(self, query, top_k = 5):
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = np.argsort(scores)[::-1][:top_k]

        results = [
            (self.documents[i], scores[i])
            for i in top_indices
        ]
        return results

In [160]:
bm25 = BM25Searcher()
bm25.build_index(remarks)

In [37]:
test_queries = [
    "house with swimming pool",
    "modern kitchen",
    "luxury bathroom",
    "near schools",
    "open floor plan",
    "large backyard",
    "chef kitchen",
    "mountain views",
    "close to shopping",
    "newly renovated"
]

In [162]:
# Generate retrieval results for both semantic search and BM25 for each query
comparison_results = []

for query in test_queries:

    semantic = searcher.search(
        query,
        top_k=5
    )

    bm25_results = bm25.search(
        query,
        top_k=5
    )


    for rank, (remark, score) in enumerate(semantic, start=1):
        comparison_results.append({
            "query": query,
            "method": "semantic",
            "rank": rank,
            "remark": remark,
            "score": float(score)
        })


    for rank, (remark, score) in enumerate(bm25_results, start=1):
        comparison_results.append({
            "query": query,
            "method": "bm25",
            "rank": rank,
            "remark": remark,
            "score": float(score)
        })

In [163]:
comparison_df = pd.DataFrame(comparison_results)
comparison_df.head()

,query,method,rank,remark,score
0,house with swimming pool,semantic,1,Wonderful opportunity to buy a pool home with ...,0.717106
1,house with swimming pool,semantic,2,Discover your dream pool home in West Lancaste...,0.630419
2,house with swimming pool,semantic,3,Welcome to your dream POOL home in the heart o...,0.607248
3,house with swimming pool,semantic,4,"San Jacinto Pool Home, built in 2004 this 4 be...",0.599817
4,house with swimming pool,semantic,5,Beautiful 4-Bedroom Home with Swimming Pool in...,0.586863


In [164]:
# Add relevance labels
comparison_df['relevant'] = None

I will manually label the relevance of these remarks to the queries to uphold any human judgement with this work. 

In [165]:
comparison_df.columns

Index(['query', 'method', 'rank', 'remark', 'score', 'relevant'], dtype='str')

In [166]:
comparison_df.to_csv("../data/comparison_results.csv", index=False)

In [167]:
comparison_df.shape

(100, 6)

In [168]:
comparison_df = pd.read_csv("../data/final_comparison_results.csv",  encoding='latin1')

In [169]:
comparison_df.shape

(100, 6)

In [170]:
comparison_df['relevant'].value_counts()

relevant
1    76
0    24
Name: count, dtype: int64

In [171]:
def precision_at_5(group):
    return (group["relevant"] > 0).sum() / 5

In [172]:
precision_results = (
    comparison_df
    .groupby(["query", "method"])
    .apply(precision_at_5)
    .reset_index(name="precision@5")
)

precision_results.head()

,query,method,precision@5
0,chef kitchen,bm25,0.0
1,chef kitchen,semantic,0.8
2,close to shopping,bm25,1.0
3,close to shopping,semantic,1.0
4,house with swimming pool,bm25,1.0


In [173]:
overall_precision = (
    precision_results
    .groupby("method")["precision@5"]
    .mean()
)

overall_precision

method
bm25        0.74
semantic    0.78
Name: precision@5, dtype: float64

Semantic search achieved an average Precision@5 of 0.78, compared with 0.74 for BM25. This indicates that semantic search retrieved a higher proportion of relevant listings among its top five results. 

In [174]:
def recall_at_5(group):
    relevant_retrieved = group["relevant"].sum()
    total_relevant = group["relevant"].sum()

    if total_relevant == 0:
        return 0

    return relevant_retrieved / total_relevant


In [175]:
recall_results = (
    comparison_df
    .groupby(["query", "method"])
    .apply(recall_at_5)
    .reset_index(name="recall@5")
)

recall_results.head()

,query,method,recall@5
0,chef kitchen,bm25,0.0
1,chef kitchen,semantic,1.0
2,close to shopping,bm25,1.0
3,close to shopping,semantic,1.0
4,house with swimming pool,bm25,1.0


In [176]:
overall_recall = (
    recall_results
    .groupby("method")["recall@5"]
    .mean()
)

overall_recall

method
bm25        0.9
semantic    1.0
Name: recall@5, dtype: float64

Deliverable 4: Semantic Search vs. BM25

Semantic search was compared with BM25 keyword search using the same evaluation queries and top-5 results. Each retrieved listing was manually labeled as relevant or not relevant, and Precision@5 and Recall@5 were then calculated to evaluate retrieval quality.

Semantic search outperformed BM25 on both metrics. It achieved a Precision@5 of 0.78 compared with 0.74 for BM25, indicating that a greater proportion of its top-five results were relevant. Semantic search also achieved 1.00 Recall@5, compared with 0.90 for BM25, indicating that it retrieved the relevant listings more consistently within the evaluated results. Overall, the results suggest that embedding-based semantic search provides better retrieval quality than traditional keyword matching for these real-estate queries.

We also completed creating a relevance evaluation mark for >= 50 query-result pairs (one query + one retrieved listing) that includes the manually relevance judgment.

Another evaluation method in which uses the EntityExtractor class. This class extracts the amentity of each listing and then use that as a ground-truth to allow automated vertification thaat retrieved listings contained the requested features.

In [177]:
entity_validation_df = df.sample(
    n = 100, 
    random_state=42
).copy().reset_index()

In [178]:
entity_validation_df.head()

,index,L_ListingID,L_Address,L_City,beds,baths,price,remarks,cleaned_remarks
0,521,1174195371,1081 Interlaken Terrace,Sunnyvale,3.0,3.0,1549000,Must See! Former model end-unit in the highly ...,Must See! Former model end-unit in the highly ...
1,737,1154290282,67105 Mission Drive 1,Cathedral City,7.0,5.0,999000,Stunning 4 bedrooms 3 baths home plus a detach...,Stunning 4 bedrooms 3 baths home plus a detach...
2,740,1157630609,645 W 9th St #424,Los Angeles,1.0,1.0,445000,Freshly renovated and hard to beat — this Sout...,Freshly renovated and hard to beat - this Sout...
3,660,1157498637,2505 Ocean Front Walk,Venice,4.0,5.0,10750000,Welcome to an extraordinarily beautiful oceanf...,Welcome to an extraordinarily beautiful oceanf...
4,411,1173668830,8086 E Loftwood,Orange,3.0,3.0,1129800,Perched in the prestigious hills of Serrano He...,Perched in the prestigious hills of Serrano He...


In [180]:
query = entity_validation_df.iloc[0]['cleaned_remarks']
semantic_results = searcher.search(query, top_k=5)
bm25_results = bm25.search(query, top_k=5)

In [181]:
# One listing from the entity validation dataframe
sample_listing = entity_validation_df.iloc[0]["cleaned_remarks"]

print(sample_listing)

Must See! Former model end-unit in the highly desirable Tasman Square community, beautifully upgraded and filled with natural light. This townhome offers a private backyard, open-concept layout, and quality finishes. The chefs kitchen features stainless steel appliances, breakfast bar, and ample cabinetry, flowing into the spacious dining and living areas. A sunny balcony, powder room, and laundry area complete the main level. Luxury vinyl flooring throughout. Upstairs features three bedrooms, including a spacious primary suite with custom walk-in closet, dual-sink vanity, and walk-in shower. Recent updates include freshly painted kitchen cabinets, bathroom vanities, and interior paint. Additional highlights include recessed lighting, Nest thermostat, central A/C, and a two-car garage with storage. Ideally located near Seven Seas Park, VTA light rail, tech shuttle stops, grocery stores, dining, and Downtown Sunnyvale. Near major tech companies including Apple, Google, Meta, and Nvidia,

In [182]:
extractor = EntityExtractor()

In [183]:
entities = extractor.extract_all(sample_listing)

print(entities)

{'bedrooms': 3, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': [{'term': 'primary suite', 'category': 'housing_layout'}, {'term': 'natural light', 'category': 'housing_features'}, {'term': 'stainless steel', 'category': 'kitchen'}, {'term': 'stainless steel appliances', 'category': 'kitchen'}, {'term': 'located near', 'category': 'location'}, {'term': 'easy access', 'category': 'housing_features'}, {'term': 'recessed lighting', 'category': 'housing_features'}, {'term': 'main level', 'category': 'parking'}, {'term': 'ideally located', 'category': 'location'}, {'term': 'kitchen features', 'category': 'kitchen'}, {'term': 'spacious primary', 'category': 'outdoor'}, {'term': 'flooring throughout', 'category': 'flooring'}, {'term': 'spacious primary suite', 'category': 'outdoor'}, {'term': 'car garage', 'category': 'parking'}, {'term': 'private backyard', 'category': 'outdoor'}, {'term': 'breakfast bar', 'category': 'kitchen'}, {'term': 'ideally located near', 'category': 'locati

The amenities become our search queries. 

In [184]:
entity_validation_df['query'] = None # add query to dataframe

In [185]:
entity_validation_df.loc[
    entity_validation_df.index[0], 
    'query'
] = 'stainless steel' # add the query to dataframe

In [186]:
def generate_query(text):
    entities = extractor.extract_all(text)

    amenities = entities["amenities"]

    if len(amenities) > 0:
        return amenities[0]["term"]

    return None

In [187]:
entity_validation_df["query"] = entity_validation_df["cleaned_remarks"].apply(
    generate_query
) # 50 listings automatically get queries

In [188]:
entity_validation_df['query'].value_counts().head()

query
primary suite      33
natural light      14
living room        13
living space        6
stainless steel     5
Name: count, dtype: int64

**Relevance Evaluation Using Entity Extraction**

To evaluate the quality of the semantic search system, the retrieved listings need to be compared against the user's search intent. The Week 5 requirement includes a relevance evaluation on 50 query-result pairs, where each returned listing is evaluated to determine whether it satisfies the original query.

Instead of manually labeling every result, the existing Week 3 EntityExtractor was reused to create a semi-automated relevance evaluation pipeline. The EntityExtractor identifies structured information from MLS listing remarks, including bedrooms, bathrooms, square footage, price mentions, and property amenities such as pools, garages, solar panels, private backyards, and other features.

The evaluation process works as follows:

1. A user query is provided to the semantic search system.

Example query: "homes with private backyard"

2. The query is converted into an embedding using the SentenceTransformer model. FAISS compares the query embedding against the stored listing embeddings and returns the most semantically similar listings.

3. Each retrieved listing is passed through the EntityExtractor. The extractor converts the unstructured listing remarks into structured attributes.

Example extracted listing attributes:

```python
{
    "bedrooms": 3,
    "bathrooms": 1,
    "amenities": [
        "private backyard",
        "stainless steel appliances",
        "fruit trees",
        "laundry area"
    ]
}

The key idea to emphasize is:

**Semantic Search answers:**  
> "Which listings are conceptually similar to this query?"

**EntityExtractor answers:**  
> "Does this listing actually contain the requested property features?"


In [189]:
entity_validation_df = entity_validation_df.drop(columns = ['remarks'])

In [191]:
entity_validation_df = entity_validation_df.dropna(
    subset=["query"]
).reset_index(drop=True)
# Remove listings where no query was generated

In [192]:
entity_validation_df.shape

(95, 9)

In [193]:
def is_relevant(remark, query, extractor):
    entities = extractor.extract_all(remark)

    extracted_terms = [
        item["term"].lower()
        for item in entities["amenities"]
    ]

    return query.lower() in extracted_terms

In [209]:
evaluation_results = []

for query_id, row in entity_validation_df.iterrows():

    query = row["query"]

    # Semantic
    semantic_results = searcher.search(query, top_k=5)

    for rank, (remark, score) in enumerate(semantic_results, start=1):

        relevant = is_relevant(
            remark,
            query,
            extractor
        )

        evaluation_results.append({
            "query_id": query_id,
            "query": query,
            "method": "Semantic",
            "rank": rank,
            "remark": remark,
            "score": float(score),
            "relevant": int(relevant)
        })

    # BM25
    bm25_results = bm25.search(query, top_k=5)

    for rank, (remark, score) in enumerate(bm25_results, start=1):

        relevant = is_relevant(
            remark,
            query,
            extractor
        )

        evaluation_results.append({
            "query_id": query_id,
            "query": query,
            "method": "BM25",
            "rank": rank,
            "remark": remark,
            "score": float(score),
            "relevant": int(relevant)
        })

In [214]:
evaluation_df = pd.DataFrame(evaluation_results)

In [215]:
evaluation_df.head()

,query_id,query,method,rank,remark,score,relevant
0,0,primary suite,Semantic,1,"Beautifully restored in 2008, this home offeri...",0.462930,1
1,0,primary suite,Semantic,2,Perched above it all in the desirable Kelseyvi...,0.433489,1
2,0,primary suite,Semantic,3,"Beautifully updated and thoughtfully designed,...",0.415732,1
3,0,primary suite,Semantic,4,"Tucked along a beautiful, tree-lined stretch o...",0.410085,1
4,0,primary suite,Semantic,5,"Fully renovated down to the studs in 2018, thi...",0.407748,0


In [216]:
def precision_at_5(group):
    return group["relevant"].sum() / 5

In [217]:
entity_precision = (
    evaluation_df
    .groupby(["query_id", "query", "method"])
    .apply(precision_at_5)
    .reset_index(name="precision@5")
)

In [218]:
entity_precision.groupby("method")["precision@5"].mean()

method
BM25        0.943158
Semantic    0.625263
Name: precision@5, dtype: float64

In [249]:
# Count all relevant listings for each query in the evaluation set
total_relevant_by_query = {}

for query in evaluation_df["query"].unique():
    total_relevant = sum(
        is_relevant(remark, query, extractor)
        for remark in df["cleaned_remarks"]
    )
    
    total_relevant_by_query[query] = total_relevant

print(total_relevant_by_query)

{'primary suite': 322, 'fruit trees': 38, 'stainless steel': 191, 'full bathroom': 94, 'easy access': 134, 'living room': 252, 'natural light': 282, 'living space': 323, 'everyday living': 185, 'car garage': 275, 'community pool': 39, 'family room': 130, 'bath home': 88, 'kitchen features': 82, 'spacious living': 93, 'home office': 150, 'conveniently located': 157, 'floor plan': 253, 'flooring throughout': 72, 'panoramic views': 51, 'laundry room': 133, 'golf course': 69, 'located within': 47}


In [255]:
def recall_at_5(group):
    relevant_retrieved = group["relevant"].sum()

    total_relevant = group["relevant"].sum()

    if total_relevant == 0:
        return 0

    return relevant_retrieved / total_relevant

In [256]:
recall_results = (
    evaluation_df
    .groupby(["query", "method"])
    .apply(recall_at_5)
    .reset_index(name="recall@5")
)

In [260]:
entity_recall = (
    evaluation_df
    .groupby(["query", "method"])
    .apply(recall_at_5)
    .reset_index(name="recall@5")
)

In [261]:
overall_recall = (
    entity_recall
    .groupby("method")["recall@5"]
    .mean()
)

print(overall_recall)

method
BM25        1.000000
Semantic    0.826087
Name: recall@5, dtype: float64


#### **Relevance Evaluation**

Two relevance-evaluation approaches were used. First, a subset of semantic and BM25 search results was **manually reviewed and labeled for relevance** based on whether each retrieved listing matched the query intent. This provided a direct human assessment of search quality. The manual evaluation produced an average Precision@5 of **0.78 for semantic search** and **0.74 for BM25**, indicating slightly better performance from semantic search on this subset.

A second evaluation used the **EntityExtractor** to provide a more systematic and scalable relevance check. Under this evaluation, BM25 achieved a Precision@5 of approximately **0.94** and Recall@5 of **1.00**, while semantic search achieved a Precision@5 of approximately **0.63** and Recall@5 of **0.83**.

The difference between the two evaluations highlights that search performance can vary depending on how relevance is defined and measured. The manual evaluation directly reflects human judgment of query relevance, while the EntityExtractor-based evaluation uses extracted property features to determine relevance.

Deliverable 5 - Latency < 100ms for 10k listings

In [6]:
load_dotenv()
def get_connection():
        conn = mysql.connector.connect(
            host=os.getenv("MYSQL_HOST"),
            user=os.getenv("MYSQL_USER"),
            password=os.getenv("MYSQL_PASSWORD"),
            database=os.getenv("MYSQL_DATABASE")
        )

        return conn

In [47]:
conn = get_connection()
cursor = conn.cursor(dictionary=True)
query = """ 
SELECT L_ListingID, L_Address, L_City, L_Keyword2 as beds, 
LM_Dec_3 as baths, L_SystemPrice as price, L_Remarks as remarks 
FROM rets_property 
WHERE L_Remarks IS NOT NULL AND LENGTH(L_Remarks) > 50 
ORDER BY RAND() LIMIT 10000
""" 
df = pd.read_sql(query, conn) 

C:\Users\mayab\AppData\Local\Temp\ipykernel_7512\666629343.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [48]:
df.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks
0,1153678732,601 S Prospect 103,Redondo Beach,1.0,1.0,514888,Welcome to coastal living in the heart of Redo...
1,1151420233,241 Arrowhead Drive,Palm Desert,3.0,4.0,1300000,"This expansive 3,102 square foot residence off..."
2,1157524909,6516 Sandy Point Court,Rancho Palos Verdes,3.0,3.0,1899000,Experience coastal luxury living in this stunn...
3,1173936710,5912 Squire Wells Way,Riverbank,3.0,2.0,489900,"Welcome to 5912 Squire Wells Way, a beautiful ..."
4,1166180778,32505 Candlewood Drive 22,Cathedral City,1.0,1.0,205000,"Welcome home to this light, bright, and spacio..."


In [50]:
cleaner = TextCleaner()

df["cleaned_remarks"] = (
    df["remarks"]
    .apply(cleaner.clean_text)
)

In [ ]:
#df.to_csv("../data/processed/cleaned_listing_10k.csv", index=False)

In [32]:
df_10k = pd.read_csv("../data/processed/cleaned_listing_10k.csv")
df_10k.shape

(10000, 8)

In [35]:
df_10k['remarks'].isna().sum()

np.int64(0)

In [36]:
remarks_10k = df_10k["cleaned_remarks"].tolist()

semantic_10k = SemanticSearcher()
semantic_10k.build_index(remarks_10k)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11358.01it/s]


Encoding 10000 listings...


In [39]:
# Test queries (same as before)
test_queries = [
    "house with swimming pool",
    "modern kitchen",
    "luxury bathroom",
    "near schools",
    "open floor plan",
    "large backyard",
    "chef kitchen",
    "mountain views",
    "close to shopping",
    "newly renovated"
]

In [40]:
latency_results = []

for query in test_queries:
    
    start = time.perf_counter()

    results = semantic_10k.search(query, top_k=5)

    end = time.perf_counter()

    latency_ms = (end - start) * 1000

    latency_results.append({
        "query": query,
        "latency_ms": latency_ms
    })

latency_df = pd.DataFrame(latency_results)

print(latency_df)

                      query  latency_ms
0  house with swimming pool     35.8553
1            modern kitchen     17.3901
2           luxury bathroom     14.3097
3              near schools     13.4717
4           open floor plan     12.0312
5            large backyard     12.5200
6              chef kitchen     13.7274
7            mountain views     12.0597
8         close to shopping     11.1032
9           newly renovated     13.1388


In [41]:
average_latency = latency_df["latency_ms"].mean()

print(f"Average search latency: {average_latency:.2f} ms")

Average search latency: 15.56 ms


In [42]:
if average_latency < 100:
    print("PASS: Latency is below 100 ms.")
else:
    print("FAIL: Latency is above 100 ms.")

PASS: Latency is below 100 ms.


In [43]:
print(f"Average: {latency_df['latency_ms'].mean():.2f} ms")
print(f"Maximum: {latency_df['latency_ms'].max():.2f} ms")
print(f"Minimum: {latency_df['latency_ms'].min():.2f} ms")

Average: 15.56 ms
Maximum: 35.86 ms
Minimum: 11.10 ms


#### **Latency Evaluation**

To evaluate the efficiency of the semantic search system, the FAISS
index was tested using 10,000 listing remarks. Ten representative
property-search queries were executed, and the retrieval latency was
measured for each query.

The average search latency was **15.56 ms**, with a maximum latency of
**35.86 ms** and a minimum of **11.10 ms**. The required performance
target was less than **100 ms** per search.

Therefore, the semantic search system successfully met the latency
requirement. The results demonstrate that FAISS can efficiently retrieve
relevant listings from a 10,000-listing dataset without exceeding the
specified response-time threshold.

In [44]:
cursor.close()
conn.close() # Close the database connection